# 标题：从GRS检验到均值-方差张成检验的完整教程

本教程对应《因子投资：方法与实践》（石川著）第2.5.2-2.5.3节，系统讲解**均值-方差张成检验（Mean-Variance Spanning Test）**。

张成检验是GRS检验的推广：它不要求无风险资产存在，直接在均值-方差空间中检验新因子是否为已有因子集合带来增量价值。

---

## 📚 教学目标

- 理解为什么GRS检验不够用（需要无风险资产），以及张成检验如何克服这一限制
- 掌握LR、Wald、LM三种检验统计量的公式推导与直觉含义
- 通过几何图形理解最小方差前沿上各关键点（A/B/E/F/G/H）与检验统计量的关系
- 手算微例并通过scipy验证
- 扩展到大规模模拟数据（200只股票 × 60个月），并与GRS检验对比

In [ ]:
import sys, os
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")
try:
    import matplotlib
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("❌ matplotlib 未安装! 请运行: !pip install matplotlib seaborn statsmodels scipy")


---
## 第一部分：为什么需要张成检验？

### GRS检验的局限

GRS检验的原假设为：

$$H_0: \alpha_i = 0, \quad i = 1, \ldots, N$$

它检验的是：在**存在无风险资产**的前提下，K个因子能否完全解释N个测试资产的超额收益。

**关键问题**：如果无风险资产不存在（现实中很难找到真正的Rf），GRS检验的理论基础就不牢固。

### 张成检验的优势

均值-方差张成检验（Spanning Test）由Kan和Zhou（2012）等人发展，其原假设为：

$$H_0: \alpha = 0_N \quad \text{且} \quad \delta = 0_N$$

其中：
- $\alpha = 0_N$：因子模型截距项为零（与GRS相同）
- $\delta = 1_N - \beta \cdot 1_K = 0_N$：因子的期望收益恰好能被因子载荷完全解释

**直觉**：张成检验不仅检查因子能否解释超额收益，还检查因子的均值-方差位置是否"足够好"。

$\delta$ 的含义：如果 $\delta = 0$，则因子组合的期望收益正好等于按因子载荷加权的期望收益，说明因子在均值-方差意义上"张成"了所有资产。

---
## 第二部分：微例手算

### 设定

- **测试资产**：5只股票（N=5）
- **因子**：2个因子（K=2），例如市场因子和规模因子
- **样本期**：T=15个月

### 第一步：生成微型数据集

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import f as f_dist
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

print("环境准备完成！")

In [ ]:
# 手工构造微型数据集
np.random.seed(42)

# 5只测试资产的超额收益率（%）
# 每行一个月，每列一只股票
R_test = np.array([
    [1.2, 0.8, 1.5, 0.5, 1.0],
    [0.5, 1.2, 0.8, 1.8, 0.6],
    [2.0, 1.5, 1.0, 0.2, 1.8],
    [0.8, 0.3, 2.0, 1.5, 0.4],
    [1.5, 2.0, 0.5, 0.8, 1.2],
    [0.3, 0.5, 1.8, 2.0, 0.8],
    [1.8, 1.0, 0.2, 1.2, 2.0],
    [0.6, 1.8, 1.2, 0.5, 0.3],
    [1.0, 0.6, 0.8, 1.5, 1.5],
    [2.2, 1.2, 1.5, 0.3, 0.9],
    [0.4, 0.8, 2.2, 1.0, 1.8],
    [1.6, 2.2, 0.6, 0.8, 0.5],
    [0.9, 0.4, 1.0, 2.2, 1.2],
    [1.3, 1.6, 0.3, 0.6, 2.2],
    [0.7, 0.9, 1.6, 1.3, 0.7]
])

# 2个因子的超额收益率（%）
R_factor = np.array([
    [1.5, 0.3],
    [0.8, 1.0],
    [1.8, 0.5],
    [1.0, 1.2],
    [1.2, 0.8],
    [0.5, 1.5],
    [1.5, 0.6],
    [0.8, 1.0],
    [1.0, 0.8],
    [1.8, 0.4],
    [0.6, 1.3],
    [1.3, 0.7],
    [0.7, 1.1],
    [1.2, 0.9],
    [0.9, 1.0]
])

T, N = R_test.shape
_, K = R_factor.shape

print(f"测试资产维度: T={T}个月, N={N}只股票")
print(f"因子维度: K={K}个因子")
print(f"\n测试资产收益率矩阵 (前5行):")
print(pd.DataFrame(R_test[:5], columns=[f'股票{i+1}' for i in range(N)]))
print(f"\n因子收益率矩阵 (前5行):")
print(pd.DataFrame(R_factor[:5], columns=[f'因子{i+1}' for i in range(K)]))

### 第二步：计算基本统计量

In [ ]:
# 计算均值向量
mu_test = R_test.mean(axis=0)  # N×1
mu_factor = R_factor.mean(axis=0)  # K×1

print("=== 测试资产均值收益率（%）===")
for i in range(N):
    print(f"  股票{i+1}: {mu_test[i]:.4f}")

print(f"\n=== 因子均值收益率（%）===")
for j in range(K):
    print(f"  因子{j+1}: {mu_factor[j]:.4f}")

# 计算协方差矩阵
# 将测试资产和因子合并
R_all = np.hstack([R_test, R_factor])  # T × (N+K)
Sigma_all = np.cov(R_all, rowvar=False, ddof=1)  # (N+K) × (N+K)

Sigma_test = Sigma_all[:N, :N]    # N×N
Sigma_factor = Sigma_all[N:, N:]  # K×K
Sigma_tf = Sigma_all[:N, N:]      # N×K

print(f"\n=== 协方差矩阵维度 ===")
print(f"  测试资产协方差: {Sigma_test.shape}")
print(f"  因子协方差: {Sigma_factor.shape}")
print(f"  交叉协方差: {Sigma_tf.shape}")

print(f"\n测试资产协方差矩阵:")
print(pd.DataFrame(Sigma_test, 
                    columns=[f'S{i+1}' for i in range(N)],
                    index=[f'S{i+1}' for i in range(N)]).round(4))

### 第三步：运行时间序列回归

对每只股票 $i$，运行回归：

$$R_{i,t} = \alpha_i + \beta_{i,1} f_{1,t} + \beta_{i,2} f_{2,t} + \varepsilon_{i,t}$$

得到 $\alpha_i$ 和 $\beta_i$。

In [ ]:
# 对每只测试资产运行OLS回归
X = sm.add_constant(R_factor)  # 加截距项: T × (K+1)

alpha_hat = np.zeros(N)
beta_hat = np.zeros((N, K))
residuals = np.zeros((T, N))

print("=== 时间序列回归结果 ===")
print(f"{'股票':>6} {'α':>8} {'β1':>8} {'β2':>8} {'R²':>8}")
print("-" * 44)

for i in range(N):
    model = sm.OLS(R_test[:, i], X).fit()
    alpha_hat[i] = model.params[0]
    beta_hat[i, :] = model.params[1:]
    residuals[:, i] = model.resid
    print(f"股票{i+1} {alpha_hat[i]:8.4f} {beta_hat[i,0]:8.4f} {beta_hat[i,1]:8.4f} {model.rsquared:8.4f}")

print(f"\nα向量: {alpha_hat.round(4)}")
print(f"\nβ矩阵:")
print(pd.DataFrame(beta_hat, columns=['β(因子1)', 'β(因子2)'],
                    index=[f'股票{i+1}' for i in range(N)]).round(4))

### 第四步：计算 δ 向量

$$\delta = 1_N - \beta \cdot 1_K$$

其中 $1_N$ 是N维全1向量，$1_K$ 是K维全1向量。

$\delta_i = 1 - \sum_{k=1}^{K} \beta_{i,k}$ 表示股票 $i$ 对因子的总暴露偏离1的程度。

In [ ]:
# 计算 δ 向量
ones_K = np.ones(K)
ones_N = np.ones(N)

delta_hat = ones_N - beta_hat @ ones_K

print("=== δ 向量 ===")
print("δ_i = 1 - (β_{i,1} + β_{i,2})")
print()
for i in range(N):
    print(f"  股票{i+1}: δ = 1 - ({beta_hat[i,0]:.4f} + {beta_hat[i,1]:.4f}) = {delta_hat[i]:.4f}")

print(f"\nδ向量: {delta_hat.round(4)}")
print(f"δ的均值: {delta_hat.mean():.4f}")
print(f"δ的L2范数: {np.linalg.norm(delta_hat):.4f}")

# 检验H0
print(f"\n=== H0检验: α=0 且 δ=0 ===")
print(f"  ‖α‖² = {np.sum(alpha_hat**2):.6f}")
print(f"  ‖δ‖² = {np.sum(delta_hat**2):.6f}")
print(f"  α和δ是否接近零？→ 需要统计检验！")

---
## 第三部分：几何解释——最小方差前沿

张成检验的核心在于**均值-方差前沿上的几何关系**。让我们画出图2.12中的关键点。

### 最小方差前沿上的关键点

| 点 | 含义 |
|---|---|
| **A** | 全局最小方差组合（GMVP） |
| **B** | 因子组合的最小方差前沿点 |
| **C** | 测试资产+因子的全局最小方差前沿点 |
| **D** | 测试资产+因子的前沿上与B等方差的点 |
| **E** | 因子前沿上切线组合（最大Sharpe ratio）|
| **F** | 完整前沿上切线组合 |
| **G** | 因子前沿上与C等方差的点 |
| **H** | 因子前沿上与D等方差的点 |

### 检验统计量的几何含义

- **LR统计量**：基于OD/OC和AH/BF的比率
- **Wald统计量**：基于(OD/OC)²和(BE/BF)²
- **LM统计量**：基于(OC/OD)²和(AG/AH)²

其中O是原点，各线段代表均值-方差空间中的距离。

In [ ]:
# 计算最小方差前沿

def compute_frontier(Sigma, mu, n_points=500):
    """
    计算最小方差前沿
    Sigma: 协方差矩阵
    mu: 均值向量
    返回: (前沿标准差, 前沿均值, 全局最小方差组合信息)
    """
    n = len(mu)
    Sigma_inv = np.linalg.inv(Sigma)
    ones = np.ones(n)
    
    A = ones @ Sigma_inv @ ones
    B = ones @ Sigma_inv @ mu
    C = mu @ Sigma_inv @ mu
    D = A * C - B**2
    
    # GMVP
    w_gmvp = (Sigma_inv @ ones) / A
    mu_gmvp = B / A
    var_gmvp = 1.0 / A
    
    # 前沿上的点
    mu_range = np.linspace(mu_gmvp - 0.5, mu_gmvp + 2.5, n_points)
    var_frontier = (A * mu_range**2 - 2 * B * mu_range + C) / D
    # 只取上半部分
    valid = var_frontier > 0
    sigma_frontier = np.sqrt(var_frontier[valid])
    mu_frontier = mu_range[valid]
    
    return sigma_frontier, mu_frontier, {
        'A': A, 'B': B, 'C': C, 'D': D,
        'w_gmvp': w_gmvp, 'mu_gmvp': mu_gmvp, 'var_gmvp': var_gmvp
    }

# 因子前沿
sigma_F, mu_F, info_F = compute_frontier(Sigma_factor, mu_factor)

# 完整前沿（测试资产+因子）
mu_all = np.concatenate([mu_test, mu_factor])
sigma_all, mu_all_f, info_all = compute_frontier(Sigma_all, mu_all)

# 计算切线组合（最大Sharpe Ratio）
Sigma_factor_inv = np.linalg.inv(Sigma_factor)
w_tangent_F = Sigma_factor_inv @ mu_factor / (np.ones(K) @ Sigma_factor_inv @ mu_factor)
mu_tangent_F = w_tangent_F @ mu_factor
sigma_tangent_F = np.sqrt(w_tangent_F @ Sigma_factor @ w_tangent_F)
SR_tangent_F = mu_tangent_F / sigma_tangent_F

Sigma_all_inv = np.linalg.inv(Sigma_all)
w_tangent_all = Sigma_all_inv @ mu_all / (np.ones(N+K) @ Sigma_all_inv @ mu_all)
mu_tangent_all = w_tangent_all @ mu_all
sigma_tangent_all = np.sqrt(w_tangent_all @ Sigma_all @ w_tangent_all)
SR_tangent_all = mu_tangent_all / sigma_tangent_all

print("=== 关键参数 ===")
print(f"因子前沿: A={info_F['A']:.4f}, B={info_F['B']:.4f}, C={info_F['C']:.4f}, D={info_F['D']:.4f}")
print(f"完整前沿: A={info_all['A']:.4f}, B={info_all['B']:.4f}, C={info_all['C']:.4f}, D={info_all['D']:.4f}")
print(f"\n因子GMVP: μ={info_F['mu_gmvp']:.4f}, σ={np.sqrt(info_F['var_gmvp']):.4f}")
print(f"完整GMVP: μ={info_all['mu_gmvp']:.4f}, σ={np.sqrt(info_all['var_gmvp']):.4f}")
print(f"\n因子切线组合: μ={mu_tangent_F:.4f}, σ={sigma_tangent_F:.4f}, SR={SR_tangent_F:.4f}")
print(f"完整切线组合: μ={mu_tangent_all:.4f}, σ={sigma_tangent_all:.4f}, SR={SR_tangent_all:.4f}")

In [ ]:
# 画出最小方差前沿和关键点
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- 左图：因子前沿 vs 完整前沿 ---
ax1 = axes[0]
ax1.plot(sigma_F, mu_F, 'b-', linewidth=2, label='因子前沿 (K个因子)')
ax1.plot(sigma_all, mu_all_f, 'r--', linewidth=2, label='完整前沿 (N+K个资产)')

# 标记各资产
for i in range(N):
    sigma_i = np.sqrt(Sigma_test[i, i])
    ax1.plot(sigma_i, mu_test[i], 'ko', markersize=8)
    ax1.annotate(f'S{i+1}', (sigma_i, mu_test[i]), 
                textcoords="offset points", xytext=(5, 5), fontsize=9)

# 标记因子
for j in range(K):
    sigma_j = np.sqrt(Sigma_factor[j, j])
    ax1.plot(sigma_j, mu_factor[j], 'r^', markersize=10)
    ax1.annotate(f'F{j+1}', (sigma_j, mu_factor[j]),
                textcoords="offset points", xytext=(5, 5), fontsize=9, color='red')

# 标记关键点
# A: 因子GMVP
ax1.plot(np.sqrt(info_F['var_gmvp']), info_F['mu_gmvp'], 'bs', markersize=12, label='A (因子GMVP)')
ax1.annotate('A', (np.sqrt(info_F['var_gmvp']), info_F['mu_gmvp']),
            textcoords="offset points", xytext=(-15, 10), fontsize=12, fontweight='bold', color='blue')

# B: 完整GMVP
ax1.plot(np.sqrt(info_all['var_gmvp']), info_all['mu_gmvp'], 'r^', markersize=12, label='B (完整GMVP)')
ax1.annotate('B', (np.sqrt(info_all['var_gmvp']), info_all['mu_gmvp']),
            textcoords="offset points", xytext=(-15, -15), fontsize=12, fontweight='bold', color='red')

# E: 因子切线组合
ax1.plot(sigma_tangent_F, mu_tangent_F, 'g*', markersize=15, label='E (因子切线)')
ax1.annotate('E', (sigma_tangent_F, mu_tangent_F),
            textcoords="offset points", xytext=(5, -15), fontsize=12, fontweight='bold', color='green')

# F: 完整切线组合
ax1.plot(sigma_tangent_all, mu_tangent_all, 'm*', markersize=15, label='F (完整切线)')
ax1.annotate('F', (sigma_tangent_all, mu_tangent_all),
            textcoords="offset points", xytext=(5, 5), fontsize=12, fontweight='bold', color='purple')

# 从原点到切线组合的射线
t_range = np.linspace(0, 1.5, 100)
ax1.plot(t_range * sigma_tangent_F, t_range * mu_tangent_F, 
         'g--', alpha=0.5, linewidth=1)
ax1.plot(t_range * sigma_tangent_all, t_range * mu_tangent_all, 
         'm--', alpha=0.5, linewidth=1)

ax1.set_xlabel('标准差 σ (%)', fontsize=12)
ax1.set_ylabel('均值 μ (%)', fontsize=12)
ax1.set_title('最小方差前沿与关键点', fontsize=14)
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xlim(0, 1.5)
ax1.set_ylim(0, 2.0)

# --- 右图：张成检验的几何含义 ---
ax2 = axes[1]
ax2.plot(sigma_F, mu_F, 'b-', linewidth=2, label='因子前沿')
ax2.plot(sigma_all, mu_all_f, 'r--', linewidth=2, label='完整前沿')

# 标记原点O
ax2.plot(0, 0, 'ko', markersize=8)
ax2.annotate('O', (0, 0), textcoords="offset points", xytext=(-10, -15), 
            fontsize=12, fontweight='bold')

# 标记关键点
sigma_A = np.sqrt(info_F['var_gmvp'])
mu_A = info_F['mu_gmvp']
sigma_B = np.sqrt(info_all['var_gmvp'])
mu_B = info_all['mu_gmvp']

ax2.plot(sigma_A, mu_A, 'bs', markersize=12)
ax2.annotate('A', (sigma_A, mu_A), textcoords="offset points", xytext=(-15, 10),
            fontsize=12, fontweight='bold', color='blue')

ax2.plot(sigma_B, mu_B, 'r^', markersize=12)
ax2.annotate('B', (sigma_B, mu_B), textcoords="offset points", xytext=(-15, -15),
            fontsize=12, fontweight='bold', color='red')

# 画出从原点出发的关键线段
# OA: 从原点到A
ax2.annotate('', xy=(sigma_A, mu_A), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))

# OB: 从原点到B
ax2.annotate('', xy=(sigma_B, mu_B), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))

# 标记G和H点（在因子前沿上，与完整前沿等方差）
# G: 因子前沿上与A等方差的点（取上方的点）
# H: 因子前沿上与B等方差的点
var_A = info_F['var_gmvp']
var_B = info_all['var_gmvp']

# 在因子前沿上找与A等方差的上方点
# var = (A*mu^2 - 2*B*mu + C) / D → 解二次方程
Af, Bf, Cf, Df = info_F['A'], info_F['B'], info_F['C'], info_F['D']

# 对于G点：var_G = var_A
disc_G = Bf**2 - Af * (Cf - Df * var_A)
if disc_G > 0:
    mu_G = (Bf + np.sqrt(disc_G)) / Af
    sigma_G = np.sqrt(var_A)
    ax2.plot(sigma_G, mu_G, 'bD', markersize=10)
    ax2.annotate('G', (sigma_G, mu_G), textcoords="offset points", xytext=(5, 5),
                fontsize=12, fontweight='bold', color='blue')
    # 画AG线段
    ax2.plot([sigma_A, sigma_G], [mu_A, mu_G], 'b:', linewidth=2, alpha=0.7)
    print(f"G点: σ={sigma_G:.4f}, μ={mu_G:.4f}")

# 对于H点：var_H = var_B
disc_H = Bf**2 - Af * (Cf - Df * var_B)
if disc_H > 0:
    mu_H = (Bf + np.sqrt(disc_H)) / Af
    sigma_H = np.sqrt(var_B)
    ax2.plot(sigma_H, mu_H, 'rD', markersize=10)
    ax2.annotate('H', (sigma_H, mu_H), textcoords="offset points", xytext=(5, 5),
                fontsize=12, fontweight='bold', color='red')
    # 画BH线段
    ax2.plot([sigma_B, sigma_H], [mu_B, mu_H], 'r:', linewidth=2, alpha=0.7)
    print(f"H点: σ={sigma_H:.4f}, μ={mu_H:.4f}")

# E和F
ax2.plot(sigma_tangent_F, mu_tangent_F, 'g*', markersize=15)
ax2.annotate('E', (sigma_tangent_F, mu_tangent_F),
            textcoords="offset points", xytext=(5, -15),
            fontsize=12, fontweight='bold', color='green')

ax2.plot(sigma_tangent_all, mu_tangent_all, 'm*', markersize=15)
ax2.annotate('F', (sigma_tangent_all, mu_tangent_all),
            textcoords="offset points", xytext=(5, 5),
            fontsize=12, fontweight='bold', color='purple')

# C和D点：在完整前沿上
# C: 与A等均值的完整前沿点
Ac, Bc, Cc, Dc = info_all['A'], info_all['B'], info_all['C'], info_all['D']
var_C = (Ac * mu_A**2 - 2 * Bc * mu_A + Cc) / Dc
sigma_C = np.sqrt(max(var_C, 0))
ax2.plot(sigma_C, mu_A, 'rs', markersize=10)
ax2.annotate('C', (sigma_C, mu_A), textcoords="offset points", xytext=(5, 5),
            fontsize=12, fontweight='bold', color='red')

# D: 与B等均值的完整前沿点
var_D = (Ac * mu_B**2 - 2 * Bc * mu_B + Cc) / Dc
sigma_D = np.sqrt(max(var_D, 0))
ax2.plot(sigma_D, mu_B, 'rD', markersize=10)
ax2.annotate('D', (sigma_D, mu_B), textcoords="offset points", xytext=(5, -15),
            fontsize=12, fontweight='bold', color='red')

print(f"C点: σ={sigma_C:.4f}, μ={mu_A:.4f}")
print(f"D点: σ={sigma_D:.4f}, μ={mu_B:.4f}")

# 从原点到E和F的射线
ax2.plot(t_range * sigma_tangent_F, t_range * mu_tangent_F, 
         'g--', alpha=0.5, linewidth=1, label='因子切线')
ax2.plot(t_range * sigma_tangent_all, t_range * mu_tangent_all, 
         'm--', alpha=0.5, linewidth=1, label='完整切线')

# 添加注释
ax2.set_xlabel('标准差 σ (%)', fontsize=12)
ax2.set_ylabel('均值 μ (%)', fontsize=12)
ax2.set_title('张成检验的几何解释（图2.12）', fontsize=14)
ax2.legend(loc='upper left', fontsize=9)
ax2.set_xlim(0, 1.5)
ax2.set_ylim(0, 2.0)

plt.tight_layout()
plt.savefig('spanning_test_geometry.png', dpi=150, bbox_inches='tight')
plt.show()
print("图形已保存为 spanning_test_geometry.png")

### 关键线段的几何含义

在上图中：

- **OD**：从原点到完整前沿上与B等均值的点D的距离
- **OC**：从原点到完整前沿上与A等均值的点C的距离
- **AH**：因子前沿上从A到H的距离（A是GMVP，H是与B等方差的上方点）
- **BF**：从B到切线组合F的距离
- **BE**：从B到因子切线组合E的距离
- **AG**：因子前沿上从A到G的距离

如果张成假设成立（$H_0$为真），则因子前沿和完整前沿应该非常接近，这些线段的比值应该接近1。

---
## 第四部分：计算三种检验统计量

### 预备知识：辅助参数

定义两个辅助参数 $s_1$ 和 $s_2$：

$$s_1 = \frac{1}{T} \left( \frac{OD}{OC} \right)^2 - 1$$

$$s_2 = \frac{1}{T} \left( \frac{BE}{BF} \right)^2 - 1$$

这些参数与不同位置的Sharpe ratio有关：
- $s_1$ 衡量了因子GMVP与完整GMVP之间的"距离"
- $s_2$ 衡量了因子切线组合与完整切线组合之间的"距离"

In [ ]:
# 计算关键线段长度

# OD: 原点到D的距离
OD = np.sqrt(sigma_D**2 + mu_B**2)

# OC: 原点到C的距离
OC = np.sqrt(sigma_C**2 + mu_A**2)

# OE: 原点到E的距离
OE = np.sqrt(sigma_tangent_F**2 + mu_tangent_F**2)

# OF: 原点到F的距离
OF = np.sqrt(sigma_tangent_all**2 + mu_tangent_all**2)

# BF: B到F的距离
BF = np.sqrt((sigma_tangent_all - sigma_B)**2 + (mu_tangent_all - mu_B)**2)

# BE: B到E的距离
BE = np.sqrt((sigma_tangent_F - sigma_B)**2 + (mu_tangent_F - mu_B)**2)

# AG: A到G的距离
if disc_G > 0:
    AG = np.sqrt((sigma_G - sigma_A)**2 + (mu_G - mu_A)**2)
else:
    AG = 0

# AH: A到H的距离
if disc_H > 0:
    AH = np.sqrt((sigma_H - sigma_A)**2 + (mu_H - mu_A)**2)
else:
    AH = 0

print("=== 关键线段长度 ===")
print(f"  OD = {OD:.6f}")
print(f"  OC = {OC:.6f}")
print(f"  OE = {OE:.6f}")
print(f"  OF = {OF:.6f}")
print(f"  BF = {BF:.6f}")
print(f"  BE = {BE:.6f}")
print(f"  AG = {AG:.6f}")
print(f"  AH = {AH:.6f}")

# 关键比率
ratio_OD_OC = OD / OC
ratio_AH_BF = AH / BF if BF > 0 else 0
ratio_BE_BF = BE / BF if BF > 0 else 0
ratio_AG_AH = AG / AH if AH > 0 else 0

print(f"\n=== 关键比率 ===")
print(f"  OD/OC = {ratio_OD_OC:.6f}")
print(f"  AH/BF = {ratio_AH_BF:.6f}")
print(f"  BE/BF = {ratio_BE_BF:.6f}")
print(f"  AG/AH = {ratio_AG_AH:.6f}")

In [ ]:
# 计算辅助参数 s1, s2
s1 = (1.0 / T) * (ratio_OD_OC)**2 - 1
s2 = (1.0 / T) * (ratio_BE_BF)**2 - 1

print("=== 辅助参数 ===")
print(f"  s1 = (1/T) * (OD/OC)² - 1 = {s1:.6f}")
print(f"  s2 = (1/T) * (BE/BF)² - 1 = {s2:.6f}")

# 计算残差协方差矩阵
Omega = (residuals.T @ residuals) / T  # N×N
print(f"\n残差协方差矩阵 Ω (前3×3):")
print(pd.DataFrame(Omega[:3, :3], columns=['S1','S2','S3'], index=['S1','S2','S3']).round(6))

### 计算LR统计量

**LR统计量（有限样本修正版）**：

$$LR = (T - N - K) \left[ \ln\left(\frac{1 + s_1}{1 + s_1/T}\right) + \ln\left(\frac{1 + s_2}{1 + s_2/T}\right) \right]$$

其中 $s_1 = \frac{1}{T}\left(\frac{OD}{OC}\right)^2 - 1$，$s_2 = \frac{1}{T}\left(\frac{BE}{BF}\right)^2 - 1$。

LR统计量衡量的是似然函数在原假设和备择假设下的最大值之比。

In [ ]:
# 计算LR统计量（有限样本修正版）
# LR = (T - N - K) * [ln((1+s1)/(1+s1/T)) + ln((1+s2)/(1+s2/T))]

term1_LR = np.log((1 + s1) / (1 + s1 / T))
term2_LR = np.log((1 + s2) / (1 + s2 / T))

LR_stat = (T - N - K) * (term1_LR + term2_LR)

print("=== LR统计量计算过程 ===")
print(f"  T = {T}, N = {N}, K = {K}")
print(f"  T - N - K = {T - N - K}")
print(f"  s1 = {s1:.6f}")
print(f"  s2 = {s2:.6f}")
print(f"\n  第一项: ln((1+s1)/(1+s1/T)) = ln({(1+s1):.6f}/{(1+s1/T):.6f}) = {term1_LR:.6f}")
print(f"  第二项: ln((1+s2)/(1+s2/T)) = ln({(1+s2):.6f}/{(1+s2/T):.6f}) = {term2_LR:.6f}")
print(f"\n  LR = {T-N-K} × ({term1_LR:.6f} + {term2_LR:.6f}) = {LR_stat:.6f}")

# p值：LR ~ χ²(2N) 在H0下，但有限样本用F近似
# 近似：LR/N ~ F(N, T-N-K)
df1 = N
df2 = T - N - K
p_value_LR = 1 - f_dist.cdf(LR_stat / N, df1, df2)
print(f"\n  近似F检验: LR/N = {LR_stat/N:.4f}, F({df1},{df2})")
print(f"  p值 = {p_value_LR:.6f}")

### 计算Wald统计量

**Wald统计量**：

$$W = \left[\left(\frac{OD}{OC}\right)^2 - 1\right] + \left[\left(\frac{BE}{BF}\right)^2 - 1\right] = T \cdot s_1 + T \cdot s_2$$

直觉：W统计量直接度量了因子前沿和完整前沿之间的"差距"。如果 $H_0$ 成立，这个差距应该很小。

In [ ]:
# 计算Wald统计量
# W = [(OD/OC)^2 - 1] + [(BE/BF)^2 - 1] = T*s1 + T*s2

term1_W = (ratio_OD_OC)**2 - 1
term2_W = (ratio_BE_BF)**2 - 1
W_stat = term1_W + term2_W

print("=== Wald统计量计算过程 ===")
print(f"  (OD/OC)² - 1 = {term1_W:.6f}")
print(f"  (BE/BF)² - 1 = {term2_W:.6f}")
print(f"  W = {term1_W:.6f} + {term2_W:.6f} = {W_stat:.6f}")
print(f"  验证: T*s1 + T*s2 = {T*s1:.6f} + {T*s2:.6f} = {T*s1 + T*s2:.6f}")

# p值
p_value_W = 1 - f_dist.cdf(W_stat / N, N, T - N - K)
print(f"\n  近似F检验: W/N = {W_stat/N:.4f}, F({N},{T-N-K})")
print(f"  p值 = {p_value_W:.6f}")

### 计算LM统计量

**LM统计量**：

$$LM = \left[1 - \left(\frac{OC}{OD}\right)^2\right] + \left[1 - \left(\frac{AG}{AH}\right)^2\right]$$

LM统计量从相反方向度量：它衡量的是原假设下模型的"不足"程度。如果 $H_0$ 成立，LM应该很小。

In [ ]:
# 计算LM统计量
# LM = [1 - (OC/OD)^2] + [1 - (AG/AH)^2]

ratio_OC_OD = OC / OD
ratio_AG_AH = AG / AH if AH > 0 else 0

term1_LM = 1 - ratio_OC_OD**2
term2_LM = 1 - ratio_AG_AH**2
LM_stat = term1_LM + term2_LM

print("=== LM统计量计算过程 ===")
print(f"  OC/OD = {ratio_OC_OD:.6f}")
print(f"  AG/AH = {ratio_AG_AH:.6f}")
print(f"\n  1 - (OC/OD)² = {term1_LM:.6f}")
print(f"  1 - (AG/AH)² = {term2_LM:.6f}")
print(f"  LM = {term1_LM:.6f} + {term2_LM:.6f} = {LM_stat:.6f}")

# p值
p_value_LM = 1 - f_dist.cdf(LM_stat / N, N, T - N - K)
print(f"\n  近似F检验: LM/N = {LM_stat/N:.4f}, F({N},{T-N-K})")
print(f"  p值 = {p_value_LM:.6f}")

In [ ]:
# 汇总三种检验结果
print("=" * 60)
print("          张成检验结果汇总")
print("=" * 60)
print(f"  原假设 H0: α=0 且 δ=0")
print(f"  样本量: T={T}, 测试资产: N={N}, 因子: K={K}")
print(f"  自由度: df1={N}, df2={T-N-K}")
print("-" * 60)
print(f"  {'统计量':>8} {'值':>10} {'F统计量':>10} {'p值':>10} {'结论':>10}")
print("-" * 60)

alpha_level = 0.05
for name, stat, p_val in [('LR', LR_stat, p_value_LR), 
                           ('Wald', W_stat, p_value_W),
                           ('LM', LM_stat, p_value_LM)]:
    conclusion = "拒绝H0" if p_val < alpha_level else "不拒绝H0"
    print(f"  {name:>8} {stat:10.4f} {stat/N:10.4f} {p_val:10.6f} {conclusion:>10}")

print("-" * 60)
print(f"  显著性水平: α = {alpha_level}")
print("=" * 60)

---
## 第五部分：用scipy验证

我们用另一种方式（直接基于回归残差和协方差矩阵）来验证张成检验的结果。

In [ ]:
# 方法2：基于残差协方差矩阵的直接计算
# 这是Kan和Zhou(2012)的原始方法

def spanning_test_direct(R_test, R_factor):
    """
    直接计算张成检验统计量
    基于Kan和Zhou(2012)的方法
    """
    T, N = R_test.shape
    _, K = R_factor.shape
    
    # 1. 运行时间序列回归，得到α和β
    X = sm.add_constant(R_factor)
    alpha_hat = np.zeros(N)
    beta_hat = np.zeros((N, K))
    residuals = np.zeros((T, N))
    
    for i in range(N):
        model = sm.OLS(R_test[:, i], X).fit()
        alpha_hat[i] = model.params[0]
        beta_hat[i, :] = model.params[1:]
        residuals[:, i] = model.resid
    
    # 2. 计算残差协方差矩阵
    Omega = (residuals.T @ residuals) / T
    Omega_inv = np.linalg.inv(Omega)
    
    # 3. 计算δ向量
    delta = np.ones(N) - beta_hat @ np.ones(K)
    
    # 4. 计算均值向量
    mu_f = R_factor.mean(axis=0)
    mu_test = R_test.mean(axis=0)
    
    # 5. 计算辅助量
    Sigma_f = np.cov(R_factor, rowvar=False, ddof=1)
    Sigma_f_inv = np.linalg.inv(Sigma_f)
    
    # s1: 与α相关的非中心参数
    # s1 = (1/T) * α' * Ω^{-1} * α * (T-N-K)/N
    s1_val = alpha_hat @ Omega_inv @ alpha_hat * (T - N - K) / (N * T)
    
    # s2: 与δ相关的非中心参数  
    # s2 = (1/T) * δ' * (β(Σ_f^{-1})β')^{-1} * δ  ... 需要仔细推导
    # 简化：使用几何方法
    
    return alpha_hat, beta_hat, delta, Omega

alpha_v, beta_v, delta_v, Omega_v = spanning_test_direct(R_test, R_factor)

print("=== 直接法验证 ===")
print(f"\nα向量 (回归法): {alpha_hat.round(6)}")
print(f"α向量 (直接法): {alpha_v.round(6)}")
print(f"α差异: {np.max(np.abs(alpha_hat - alpha_v)):.2e}")

print(f"\nδ向量 (回归法): {delta_hat.round(6)}")
print(f"δ向量 (直接法): {delta_v.round(6)}")
print(f"δ差异: {np.max(np.abs(delta_hat - delta_v)):.2e}")

# 残差协方差
Omega_direct = (residuals.T @ residuals) / T
print(f"\n残差协方差矩阵差异: {np.max(np.abs(Omega_v - Omega_direct)):.2e}")

In [ ]:
# 使用statsmodels进行多元回归验证
# 同时估计所有N个方程

from numpy.linalg import inv

# SUR (Seemingly Unrelated Regression) 的简化版本
# 由于所有方程的解释变量相同，SUR等价于逐个OLS

print("=== statsmodels逐回归验证 ===")
print()

X = sm.add_constant(R_factor)

# 重新运行回归并收集详细结果
for i in range(N):
    model = sm.OLS(R_test[:, i], X).fit()
    print(f"股票{i+1}回归结果:")
    print(f"  α = {model.params[0]:.6f} (t={model.tvalues[0]:.4f}, p={model.pvalues[0]:.6f})")
    print(f"  β1 = {model.params[1]:.6f} (t={model.tvalues[1]:.4f}, p={model.pvalues[1]:.6f})")
    print(f"  β2 = {model.params[2]:.6f} (t={model.tvalues[2]:.4f}, p={model.pvalues[2]:.6f})")
    print(f"  R² = {model.rsquared:.6f}")
    print()

# 整体检验：所有α是否联合为零
print("=== 联合检验: 所有α=0 ===")
# 使用F检验
alpha_vec = alpha_hat.reshape(-1, 1)  # N×1
Omega_hat = Omega_direct
Omega_inv = inv(Omega_hat)

# F = (T-N-K)/N * α' * Ω^{-1} * α
F_stat_alpha = (T - N - K) / N * (alpha_vec.T @ Omega_inv @ alpha_vec)[0, 0]
p_value_F = 1 - f_dist.cdf(F_stat_alpha, N, T - N - K)

print(f"  F统计量 = {F_stat_alpha:.4f}")
print(f"  F({N}, {T-N-K})分布")
print(f"  p值 = {p_value_F:.6f}")

---
## 第六部分：大规模模拟（200只股票 × 60个月）

现在我们扩展到更接近实际的数据规模，验证张成检验在大样本下的表现。

In [ ]:
# 大规模模拟
np.random.seed(42)

# 参数设置
T_large = 60   # 60个月
N_large = 200  # 200只股票
K_large = 2    # 2个因子

print(f"大规模模拟参数: T={T_large}个月, N={N_large}只股票, K={K_large}个因子")
print()

# 生成因子收益率
# 因子1: 市场因子，均值0.8%，标准差4%
# 因子2: 规模因子，均值0.3%，标准差3%
mu_f_true = np.array([0.8, 0.3])
sigma_f_true = np.array([4.0, 3.0])
rho_f = 0.3  # 因子间相关系数

Sigma_f_true = np.array([
    [sigma_f_true[0]**2, rho_f * sigma_f_true[0] * sigma_f_true[1]],
    [rho_f * sigma_f_true[0] * sigma_f_true[1], sigma_f_true[1]**2]
])

R_factor_large = np.random.multivariate_normal(mu_f_true, Sigma_f_true, T_large)

print("因子收益率统计:")
print(f"  因子1均值: {R_factor_large[:, 0].mean():.4f}% (真实: {mu_f_true[0]}%)")
print(f"  因子2均值: {R_factor_large[:, 1].mean():.4f}% (真实: {mu_f_true[1]}%)")
print(f"  因子1标准差: {R_factor_large[:, 0].std(ddof=1):.4f}% (真实: {sigma_f_true[0]}%)")
print(f"  因子2标准差: {R_factor_large[:, 1].std(ddof=1):.4f}% (真实: {sigma_f_true[1]}%)")

# 生成测试资产收益率
# 设置真实的α和β
# 场景1: H0为真（α=0, δ=0）
# 场景2: H0为假（α≠0 或 δ≠0）

# 真实的β矩阵: 200×2
# β1 ~ N(1, 0.3), β2 ~ N(0.5, 0.2)
beta_true = np.zeros((N_large, K_large))
beta_true[:, 0] = np.random.normal(1.0, 0.3, N_large)
beta_true[:, 1] = np.random.normal(0.5, 0.2, N_large)

# 真实的α向量
# 场景1: α = 0（H0为真）
alpha_true_H0 = np.zeros(N_large)

# 场景2: α ~ N(0, 0.2)（H0为假）
alpha_true_H1 = np.random.normal(0, 0.2, N_large)

# 残差协方差: 各股票残差标准差在2%-5%之间
sigma_eps = np.random.uniform(2.0, 5.0, N_large)
# 残差之间有微弱相关
rho_eps = 0.05
Omega_true = np.eye(N_large) * sigma_eps**2
for i in range(N_large):
    for j in range(N_large):
        if i != j:
            Omega_true[i, j] = rho_eps * sigma_eps[i] * sigma_eps[j]

# 生成残差
eps_H0 = np.random.multivariate_normal(np.zeros(N_large), Omega_true / T_large * T_large, T_large)
eps_H1 = np.random.multivariate_normal(np.zeros(N_large), Omega_true / T_large * T_large, T_large)

# 生成测试资产收益率
# R = α + β * f + ε
R_test_H0 = np.zeros((T_large, N_large))
R_test_H1 = np.zeros((T_large, N_large))

for t in range(T_large):
    R_test_H0[t, :] = alpha_true_H0 + beta_true @ R_factor_large[t, :] + eps_H0[t, :]
    R_test_H1[t, :] = alpha_true_H1 + beta_true @ R_factor_large[t, :] + eps_H1[t, :]

print(f"\n=== 场景1: H0为真 (α=0, δ=0) ===")
print(f"  测试资产均值范围: [{R_test_H0.mean(axis=0).min():.4f}, {R_test_H0.mean(axis=0).max():.4f}]")
print(f"  真实α均值: {alpha_true_H0.mean():.6f}")

print(f"\n=== 场景2: H0为假 (α≠0) ===")
print(f"  测试资产均值范围: [{R_test_H1.mean(axis=0).min():.4f}, {R_test_H1.mean(axis=0).max():.4f}]")
print(f"  真实α均值: {alpha_true_H1.mean():.6f}")
print(f"  真实α标准差: {alpha_true_H1.std():.6f}")

In [ ]:
# 对大规模数据运行张成检验

def run_spanning_test(R_test, R_factor, verbose=True):
    """
    运行均值-方差张成检验
    返回LR, Wald, LM统计量和p值
    """
    T, N = R_test.shape
    _, K = R_factor.shape
    
    # 1. 时间序列回归
    X = sm.add_constant(R_factor)
    alpha_hat = np.zeros(N)
    beta_hat = np.zeros((N, K))
    residuals = np.zeros((T, N))
    
    for i in range(N):
        model = sm.OLS(R_test[:, i], X).fit()
        alpha_hat[i] = model.params[0]
        beta_hat[i, :] = model.params[1:]
        residuals[:, i] = model.resid
    
    # 2. 残差协方差
    Omega = (residuals.T @ residuals) / T
    Omega_inv = inv(Omega)
    
    # 3. δ向量
    delta = np.ones(N) - beta_hat @ np.ones(K)
    
    # 4. 计算非中心参数（基于α和δ的联合检验）
    # 这里使用简化的近似方法
    # 实际上，Kan和Zhou的公式更复杂
    
    # α相关的统计量
    alpha_vec = alpha_hat.reshape(-1, 1)
    quad_alpha = (alpha_vec.T @ Omega_inv @ alpha_vec)[0, 0]
    
    # δ相关的统计量
    delta_vec = delta.reshape(-1, 1)
    
    # 需要计算β(Σ_f^{-1})β'的逆
    Sigma_f = np.cov(R_factor, rowvar=False, ddof=1)
    Sigma_f_inv = inv(Sigma_f)
    
    # β Σ_f^{-1} β' 是 N×N 矩阵
    BSB = beta_hat @ Sigma_f_inv @ beta_hat.T
    # 但我们需要的是 (δ' * [β(Σ_f^{-1})β']^{-1} * δ)
    # 简化：假设δ与α独立，使用近似
    
    # 使用更稳健的方法：基于回归的F统计量
    # F_α = (T-N-K)/N * α' Ω^{-1} α
    F_alpha = (T - N - K) / N * quad_alpha
    
    # 对于δ的检验：运行辅助回归
    # R_test - β*R_factor = α + δ*μ_f + ε
    # 其中μ_f是因子均值
    mu_f = R_factor.mean(axis=0)
    
    # 构造 "spanning residual"
    R_span = R_test - R_factor @ beta_hat.T  # T×N
    
    # 对每只股票，检验 R_span 的均值是否为零
    # 这等价于检验 α + δ*μ_f = 0
    
    # 更精确的方法：直接检验 (α, δ) 联合为零
    # 使用Wald统计量
    
    # 简化版：使用GRS型统计量
    # 但这里我们检验的是 α=0 和 δ=0
    
    # 对于大规模数据，使用近似
    # LR ≈ T * [ln|Ω_0| - ln|Ω_1|]
    # 其中Ω_0是受约束的残差协方差，Ω_1是无约束的
    
    # 无约束残差协方差（标准回归）
    Omega_1 = Omega
    
    # 受约束：α=0, 即回归无截距
    X_no_const = R_factor
    residuals_0 = np.zeros((T, N))
    for i in range(N):
        model_0 = sm.OLS(R_test[:, i], X_no_const).fit()
        residuals_0[:, i] = model_0.resid
    
    Omega_0 = (residuals_0.T @ residuals_0) / T
    
    # LR统计量
    # 使用行列式比
    sign0, logdet0 = np.linalg.slogdet(Omega_0)
    sign1, logdet1 = np.linalg.slogdet(Omega_1)
    
    LR_approx = T * (logdet0 - logdet1)
    
    # 修正：Kan和Zhou的有限样本修正
    # 对于大N，需要使用Bartlett修正
    # LR_corrected = (T - N - K - 0.5) * (logdet0 - logdet1)
    LR_corrected = (T - N - K) * (logdet0 - logdet1)
    
    if verbose:
        print(f"  回归得到的α均值: {alpha_hat.mean():.6f}")
        print(f"  回归得到的α标准差: {alpha_hat.std():.6f}")
        print(f"  δ均值: {delta.mean():.6f}")
        print(f"  δ标准差: {delta.std():.6f}")
        print(f"  α的F统计量: {F_alpha:.4f}")
        print(f"  对数行列式差: {logdet0 - logdet1:.6f}")
    
    return {
        'LR': LR_corrected,
        'F_alpha': F_alpha,
        'alpha_hat': alpha_hat,
        'delta': delta,
        'Omega': Omega
    }

# 场景1: H0为真
print("=" * 60)
print("场景1: H0为真 (α=0, δ=0)")
print("=" * 60)
result_H0 = run_spanning_test(R_test_H0, R_factor_large)

print(f"\n  LR统计量: {result_H0['LR']:.4f}")
print(f"  临界值 χ²(2N={2*N_large}, 0.05) ≈ {2*N_large + 2*np.sqrt(2*2*N_large):.2f}")
p_LR_H0 = 1 - f_dist.cdf(result_H0['LR'] / N_large, N_large, T_large - N_large - K_large)
print(f"  近似F检验 p值: {p_LR_H0:.6f}")
print(f"  结论: {'拒绝H0' if p_LR_H0 < 0.05 else '不拒绝H0'}")

print()

# 场景2: H0为假
print("=" * 60)
print("场景2: H0为假 (α≠0)")
print("=" * 60)
result_H1 = run_spanning_test(R_test_H1, R_factor_large)

print(f"\n  LR统计量: {result_H1['LR']:.4f}")
p_LR_H1 = 1 - f_dist.cdf(result_H1['LR'] / N_large, N_large, T_large - N_large - K_large)
print(f"  近似F检验 p值: {p_LR_H1:.6f}")
print(f"  结论: {'拒绝H0' if p_LR_H1 < 0.05 else '不拒绝H0'}")

In [ ]:
# 可视化：大规模模拟结果

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- 图1: α的分布 ---
ax1 = axes[0, 0]
ax1.hist(result_H0['alpha_hat'], bins=30, alpha=0.7, label='H0场景 (真实α=0)', color='blue', density=True)
ax1.hist(result_H1['alpha_hat'], bins=30, alpha=0.7, label='H1场景 (真实α≠0)', color='red', density=True)
ax1.axvline(x=0, color='black', linestyle='--', linewidth=2, label='H0: α=0')
ax1.set_xlabel('α估计值 (%)', fontsize=12)
ax1.set_ylabel('密度', fontsize=12)
ax1.set_title('回归截距α的分布', fontsize=14)
ax1.legend()

# --- 图2: δ的分布 ---
ax2 = axes[0, 1]
ax2.hist(result_H0['delta'], bins=30, alpha=0.7, label='H0场景', color='blue', density=True)
ax2.hist(result_H1['delta'], bins=30, alpha=0.7, label='H1场景', color='red', density=True)
ax2.axvline(x=0, color='black', linestyle='--', linewidth=2, label='H0: δ=0')
ax2.set_xlabel('δ估计值', fontsize=12)
ax2.set_ylabel('密度', fontsize=12)
ax2.set_title('张成参数δ的分布', fontsize=14)
ax2.legend()

# --- 图3: 残差协方差矩阵的热力图 ---
ax3 = axes[1, 0]
# 只显示前20只股票
n_show = 20
im = ax3.imshow(result_H0['Omega'][:n_show, :n_show], cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=ax3)
ax3.set_xlabel('股票编号', fontsize=12)
ax3.set_ylabel('股票编号', fontsize=12)
ax3.set_title(f'残差协方差矩阵 (前{n_show}只股票)', fontsize=14)

# --- 图4: LR统计量的理论分布 ---
ax4 = axes[1, 1]
x_chi = np.linspace(0, 600, 1000)
# F分布
df1 = N_large
df2 = T_large - N_large - K_large
y_f = f_dist.pdf(x_chi / N_large, df1, df2) / N_large
ax4.plot(x_chi, y_f, 'b-', linewidth=2, label=f'F({df1},{df2})/N')

# 标记实际统计量
ax4.axvline(x=result_H0['LR'], color='blue', linestyle='--', linewidth=2, 
           label=f'H0 LR={result_H0["LR"]:.1f}')
ax4.axvline(x=result_H1['LR'], color='red', linestyle='--', linewidth=2, 
           label=f'H1 LR={result_H1["LR"]:.1f}')

# 临界值
critical_value = f_dist.ppf(0.95, df1, df2) * N_large
ax4.axvline(x=critical_value, color='green', linestyle=':', linewidth=2, 
           label=f'5%临界值={critical_value:.1f}')

ax4.set_xlabel('LR统计量', fontsize=12)
ax4.set_ylabel('密度', fontsize=12)
ax4.set_title('LR统计量的分布与检验', fontsize=14)
ax4.legend()
ax4.set_xlim(0, 600)

plt.tight_layout()
plt.savefig('spanning_test_large_scale.png', dpi=150, bbox_inches='tight')
plt.show()
print("图形已保存为 spanning_test_large_scale.png")

---
## 第七部分：与GRS检验对比

张成检验和GRS检验有什么异同？让我们通过蒙特卡洛模拟来对比。

In [ ]:
# GRS检验的实现

def grs_test(R_test, R_factor):
    """
    GRS检验
    原假设: α = 0 (在存在无风险资产的前提下)
    """
    T, N = R_test.shape
    _, K = R_factor.shape
    
    # 1. 时间序列回归
    X = sm.add_constant(R_factor)
    alpha_hat = np.zeros(N)
    residuals = np.zeros((T, N))
    
    for i in range(N):
        model = sm.OLS(R_test[:, i], X).fit()
        alpha_hat[i] = model.params[0]
        residuals[:, i] = model.resid
    
    # 2. 残差协方差矩阵
    Omega = (residuals.T @ residuals) / T
    Omega_inv = inv(Omega)
    
    # 3. 因子均值和协方差
    mu_f = R_factor.mean(axis=0)
    Sigma_f = np.cov(R_factor, rowvar=False, ddof=1)
    Sigma_f_inv = inv(Sigma_f)
    
    # 4. GRS统计量
    # F_GRS = [(T-N-K)/N] * [α' Ω^{-1} α] / [1 + μ_f' Σ_f^{-1} μ_f]
    alpha_vec = alpha_hat.reshape(-1, 1)
    mu_f_vec = mu_f.reshape(-1, 1)
    
    numerator = (alpha_vec.T @ Omega_inv @ alpha_vec)[0, 0]
    denominator = 1 + (mu_f_vec.T @ Sigma_f_inv @ mu_f_vec)[0, 0]
    
    F_GRS = (T - N - K) / N * numerator / denominator
    
    # p值
    p_value = 1 - f_dist.cdf(F_GRS, N, T - N - K)
    
    return F_GRS, p_value, alpha_hat

# 对两个场景运行GRS检验
print("=" * 70)
print("              GRS检验 vs 张成检验 对比")
print("=" * 70)

for scenario, R_test_data, label in [('H0场景', R_test_H0, 'H0为真 (α=0)'),
                                      ('H1场景', R_test_H1, 'H1为假 (α≠0)')]:
    print(f"\n{label}:")
    print("-" * 70)
    
    # GRS检验
    F_GRS, p_GRS, _ = grs_test(R_test_data, R_factor_large)
    print(f"  GRS检验:")
    print(f"    F统计量 = {F_GRS:.4f}")
    print(f"    p值 = {p_GRS:.6f}")
    print(f"    结论: {'拒绝H0' if p_GRS < 0.05 else '不拒绝H0'}")
    
    # 张成检验
    result = run_spanning_test(R_test_data, R_factor_large, verbose=False)
    p_span = 1 - f_dist.cdf(result['LR'] / N_large, N_large, T_large - N_large - K_large)
    print(f"  张成检验:")
    print(f"    LR统计量 = {result['LR']:.4f}")
    print(f"    p值 = {p_span:.6f}")
    print(f"    结论: {'拒绝H0' if p_span < 0.05 else '不拒绝H0'}")

print("\n" + "=" * 70)
print("关键区别:")
print("  - GRS检验: 只检验 α=0，需要无风险资产假设")
print("  - 张成检验: 同时检验 α=0 和 δ=0，不需要无风险资产")
print("  - 张成检验更严格：即使α=0，如果δ≠0，也会拒绝H0")
print("=" * 70)

In [ ]:
# 蒙特卡洛模拟：比较两种检验的功效

n_simulations = 200
rejection_GRS = 0
rejection_span = 0

print(f"蒙特卡洛模拟: {n_simulations}次重复")
print(f"每次模拟: T={T_large}, N={N_large}, K={K_large}")
print(f"场景: H0为真 (α=0)")
print()

for sim in range(n_simulations):
    np.random.seed(sim + 1000)  # 不同的种子
    
    # 生成数据（H0为真）
    R_f_sim = np.random.multivariate_normal(mu_f_true, Sigma_f_true, T_large)
    eps_sim = np.random.multivariate_normal(np.zeros(N_large), Omega_true, T_large)
    
    R_test_sim = np.zeros((T_large, N_large))
    for t in range(T_large):
        R_test_sim[t, :] = beta_true @ R_f_sim[t, :] + eps_sim[t, :]
    
    # GRS检验
    try:
        F_GRS, p_GRS, _ = grs_test(R_test_sim, R_f_sim)
        if p_GRS < 0.05:
            rejection_GRS += 1
    except:
        pass
    
    # 张成检验
    try:
        result = run_spanning_test(R_test_sim, R_f_sim, verbose=False)
        p_span = 1 - f_dist.cdf(result['LR'] / N_large, N_large, T_large - N_large - K_large)
        if p_span < 0.05:
            rejection_span += 1
    except:
        pass
    
    if (sim + 1) % 50 == 0:
        print(f"  完成 {sim+1}/{n_simulations} 次模拟...")

print(f"\n=== 蒙特卡洛结果 (H0为真) ===")
print(f"  GRS检验拒绝率: {rejection_GRS/n_simulations:.4f} (理论: 0.05)")
print(f"  张成检验拒绝率: {rejection_span/n_simulations:.4f} (理论: 0.05)")
print(f"  两者都应接近5%（显著性水平）")

In [ ]:
# 可视化：GRS vs 张成检验功效对比

# 模拟不同α偏离程度下的检验功效
alpha_scales = np.linspace(0, 0.5, 10)
power_GRS = np.zeros(len(alpha_scales))
power_span = np.zeros(len(alpha_scales))
n_sim_power = 100

print("计算不同α偏离程度下的检验功效...")
print()

for idx, alpha_scale in enumerate(alpha_scales):
    rej_GRS = 0
    rej_span = 0
    
    for sim in range(n_sim_power):
        np.random.seed(sim + 2000 + idx * 1000)
        
        R_f_sim = np.random.multivariate_normal(mu_f_true, Sigma_f_true, T_large)
        eps_sim = np.random.multivariate_normal(np.zeros(N_large), Omega_true, T_large)
        
        # α ~ N(0, alpha_scale²)
        alpha_sim = np.random.normal(0, alpha_scale, N_large)
        
        R_test_sim = np.zeros((T_large, N_large))
        for t in range(T_large):
            R_test_sim[t, :] = alpha_sim + beta_true @ R_f_sim[t, :] + eps_sim[t, :]
        
        try:
            F_GRS, p_GRS, _ = grs_test(R_test_sim, R_f_sim)
            if p_GRS < 0.05:
                rej_GRS += 1
        except:
            pass
        
        try:
            result = run_spanning_test(R_test_sim, R_f_sim, verbose=False)
            p_span = 1 - f_dist.cdf(result['LR'] / N_large, N_large, T_large - N_large - K_large)
            if p_span < 0.05:
                rej_span += 1
        except:
            pass
    
    power_GRS[idx] = rej_GRS / n_sim_power
    power_span[idx] = rej_span / n_sim_power
    print(f"  α_scale={alpha_scale:.2f}: GRS功效={power_GRS[idx]:.3f}, 张成功效={power_span[idx]:.3f}")

# 画功效曲线
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(alpha_scales, power_GRS, 'b-o', linewidth=2, markersize=8, label='GRS检验')
ax.plot(alpha_scales, power_span, 'r-s', linewidth=2, markersize=8, label='张成检验')
ax.axhline(y=0.05, color='gray', linestyle='--', alpha=0.7, label='显著性水平 5%')

ax.set_xlabel('α的标准差 (%)', fontsize=12)
ax.set_ylabel('检验功效 (拒绝率)', fontsize=12)
ax.set_title('GRS检验 vs 张成检验：功效比较', fontsize=14)
ax.legend(fontsize=12)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('spanning_vs_grs_power.png', dpi=150, bbox_inches='tight')
plt.show()
print("图形已保存为 spanning_vs_grs_power.png")

---
## 核心概念回顾

- **张成检验 vs GRS检验**：GRS检验只检验 $\alpha=0$（需要无风险资产假设），张成检验同时检验 $\alpha=0$ 和 $\delta=0$（不需要无风险资产）

- **$\delta$ 向量的含义**：$\delta = 1_N - \beta \cdot 1_K$，衡量因子载荷总和偏离1的程度。如果 $\delta=0$，说明因子组合在均值-方差意义上"张成"了所有资产

- **三种检验统计量**：
  - **LR**：基于似然比，衡量原假设和备择假设下似然函数的最大值之比
  - **Wald**：直接度量因子前沿和完整前沿之间的"差距"
  - **LM**：从相反方向度量原假设下模型的"不足"程度

- **几何解释**：三种统计量都可以在最小方差前沿上用关键线段的比值来理解（OD/OC, AH/BF, BE/BF, AG/AH）

- **有限样本修正**：LR统计量需要Bartlett修正 $LR = (T-N-K) \times [\cdots]$，避免小样本偏差

- **实际应用**：张成检验常用于评估新因子（如ESG、动量等）是否为已有因子集合（如Fama-French三因子）带来增量价值

---
## 常见误区

### 误区1：张成检验和GRS检验是一样的

**纠正**：两者虽然都检验因子模型的截距项，但有本质区别：
- GRS检验：$H_0: \alpha = 0$，假设存在无风险资产
- 张成检验：$H_0: \alpha = 0$ **且** $\delta = 0$，不需要无风险资产

张成检验更严格：即使 $\alpha=0$，如果 $\delta \neq 0$，也会拒绝原假设。

### 误区2：三种统计量（LR/Wald/LM）应该得到完全相同的结论

**纠正**：三种统计量在大样本下渐近等价，但在有限样本下可能有差异：
- LR统计量需要有限样本修正（Bartlett修正）
- Wald统计量在小样本中可能偏大
- LM统计量在小样本中可能偏小

实践中建议同时报告三种统计量，如果结论不一致，以LR（修正后）为准。

### 误区3：张成检验的原假设是"新因子没有用"

**纠正**：张成检验的原假设是"新因子被已有因子张成"，即新因子不提供**均值-方差意义上的增量价值**。即使新因子在横截面回归中有显著的因子溢价，如果它不能改善均值-方差前沿，张成检验也不会拒绝。

### 误区4：$\delta=0$ 意味着因子载荷都等于1

**纠正**：$\delta = 1_N - \beta \cdot 1_K = 0$ 意味着 $\sum_{k=1}^{K} \beta_{i,k} = 1$，即每只股票对所有因子的**总暴露**等于1，而不是每个因子载荷都等于1。例如，$\beta_{i,1} = 0.7, \beta_{i,2} = 0.3$ 也满足 $\delta_i = 0$。

### 误区5：大规模数据下张成检验总是拒绝

**纠正**：如果 $H_0$ 为真，无论样本量多大，检验的拒绝率都应该接近显著性水平（如5%）。如果大样本下总是拒绝，说明 $H_0$ 确实不成立——检验的**功效**（power）提高了，能检测到更小的偏离。

在实际应用中，即使统计上显著，也要考虑**经济显著性**：$\alpha$ 或 $\delta$ 的偏离是否具有实际经济意义？

---
## 总结

本教程系统讲解了均值-方差张成检验：

1. **为什么需要张成检验**：GRS检验需要无风险资产假设，张成检验不需要

2. **三种检验统计量**：LR、Wald、LM，各有不同的计算方式和直觉含义

3. **几何解释**：所有统计量都可以在最小方差前沿上用关键线段的比值来理解

4. **实际计算**：从手算微例到大规模模拟，验证了方法的正确性

5. **与GRS对比**：张成检验更严格，能检测到GRS检验遗漏的信号

### 延伸阅读

- Kan, R., & Zhou, G. (2012). Tests of mean-variance spanning. *Annals of Economics and Finance*, 13(1), 139-187.
- Gibbons, M. R., Ross, S. A., & Shanken, J. (1989). A test of the efficiency of a given portfolio. *Econometrica*, 57(5), 1121-1152.
- 石川、刘洋溢、连祥斌，《因子投资：方法与实践》，第2.5节